# Hybrid Robotic Vision: Notebook 4 — Classification and Semantic-Gain Metrics

This notebook computes the **semantic-gain metrics** for the hybrid visual telemetry experiment.

It compares two inputs for the **same selected ROI**:

1. the **compressed-video crop** from the base stream  
2. the **decoded JPEG still** reconstructed from the ROI side channel

The goal is not yet to claim final classification accuracy against trusted labels.  
Instead, this notebook measures whether the still-image side channel provides **stronger semantic evidence** than the compressed-video crop.

## Metrics produced in this notebook

### Coverage-support metrics
- `num_classified_rois`

### Semantic-gain metrics
- `mean_video_conf`
- `mean_still_conf`
- `mean_conf_gain`
- `median_conf_gain`
- `positive_conf_gain_rate`
- `pred_change_rate`
- `small_object_mean_conf_gain`

### Optional uncertainty metric
- `mean_entropy_gain`

These metrics feed directly into the next aggregation notebook, which will combine:
- base-video bitrate
- ROI still bitrate
- semantic gain

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Define paths and select the experiment run

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/hybrid_robotic_vision")
RUNS = ROOT / "runs"

SEQUENCE_NAME = "M0101"   # <-- EDIT THIS
REGIME = "low"            # <-- choose: "low" or "moderate"

INPUT_DIR = RUNS / "roi_still_compression_jpeg" / SEQUENCE_NAME / REGIME / "manifests"
ROI_STILL_MANIFEST_CSV = INPUT_DIR / "roi_still_manifest.csv"
ROI_STILL_SUMMARY_JSON = INPUT_DIR / "run_summary.json"

OUT_DIR = RUNS / "classification_semantic_gain" / SEQUENCE_NAME / REGIME / "manifests"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input manifest:", ROI_STILL_MANIFEST_CSV)
print("Input summary:", ROI_STILL_SUMMARY_JSON)
print("Output dir:", OUT_DIR)

## 3. Install packages

In [ ]:
!pip install -q torch torchvision pandas tqdm pillow matplotlib

## 4. Imports

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image

import torch
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights
import torch.nn.functional as F

## 5. Load the ROI still manifest

In [ ]:
if not ROI_STILL_MANIFEST_CSV.exists():
    raise FileNotFoundError(f"ROI still manifest not found: {ROI_STILL_MANIFEST_CSV}")

roi_still_df = pd.read_csv(ROI_STILL_MANIFEST_CSV)
print(f"Loaded ROI still manifest with {len(roi_still_df)} rows")
roi_still_df.head()

## 6. Configure the evaluation subset

In [ ]:
MAX_ROWS = None   # Example: 100
USE_ONLY_SUCCESSFUL_STILLS = True

work_df = roi_still_df.copy()

if USE_ONLY_SUCCESSFUL_STILLS and "still_decode_ok" in work_df.columns:
    work_df = work_df[work_df["still_decode_ok"] == True].copy()

if MAX_ROWS is not None:
    work_df = work_df.head(MAX_ROWS).copy()

print(f"Using {len(work_df)} rows for classification")
work_df.head()

## 7. Load the classifier

We use a standard ImageNet-pretrained ResNet-50 as a stable semantic probe.

This does **not** mean the predictions are UAV-task labels.  
The goal here is relative comparison:
- compressed-video crop vs decoded still crop
- confidence and entropy change

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)
model.eval()
model.to(device)

preprocess = weights.transforms()
imagenet_categories = weights.meta["categories"]

print("Loaded ResNet-50 with", len(imagenet_categories), "categories")

## 8. Define inference helpers

In [ ]:
def predict_image(image_path):
    image_path = Path(image_path)
    if not image_path.exists():
        return None

    img = Image.open(image_path).convert("RGB")
    x = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]

    top1_idx = int(np.argmax(probs))
    top1_conf = float(probs[top1_idx])
    top1_label = imagenet_categories[top1_idx]

    eps = 1e-12
    entropy = float(-(probs * np.log(probs + eps)).sum())

    top5_idx = np.argsort(probs)[-5:][::-1]
    top5 = [
        {
            "idx": int(i),
            "label": imagenet_categories[int(i)],
            "conf": float(probs[int(i)])
        }
        for i in top5_idx
    ]

    return {
        "top1_idx": top1_idx,
        "top1_label": top1_label,
        "top1_conf": top1_conf,
        "entropy": entropy,
        "top5": top5,
    }

## 9. Run classification on both branches

In [ ]:
results_rows = []

required_cols = ["comp_crop_path", "decoded_still_path"]
for col in required_cols:
    if col not in work_df.columns:
        raise KeyError(f"Required column missing from manifest: {col}")

for _, row in tqdm(work_df.iterrows(), total=len(work_df), desc="Classifying ROI pairs"):
    comp_path = row["comp_crop_path"]
    still_path = row["decoded_still_path"]

    if not isinstance(comp_path, str) or not Path(comp_path).exists():
        continue
    if not isinstance(still_path, str) or not Path(still_path).exists():
        continue

    comp_pred = predict_image(comp_path)
    still_pred = predict_image(still_path)

    if comp_pred is None or still_pred is None:
        continue

    conf_gain = still_pred["top1_conf"] - comp_pred["top1_conf"]
    entropy_gain = comp_pred["entropy"] - still_pred["entropy"]
    pred_changed = int(comp_pred["top1_idx"] != still_pred["top1_idx"])
    positive_conf_gain = int(conf_gain > 0)

    results_rows.append({
        **row.to_dict(),
        "video_pred_idx": comp_pred["top1_idx"],
        "video_pred_label": comp_pred["top1_label"],
        "video_pred_conf": comp_pred["top1_conf"],
        "video_entropy": comp_pred["entropy"],
        "video_top5_json": json.dumps(comp_pred["top5"]),
        "still_pred_idx": still_pred["top1_idx"],
        "still_pred_label": still_pred["top1_label"],
        "still_pred_conf": still_pred["top1_conf"],
        "still_entropy": still_pred["entropy"],
        "still_top5_json": json.dumps(still_pred["top5"]),
        "conf_gain": conf_gain,
        "entropy_gain": entropy_gain,
        "pred_changed": pred_changed,
        "positive_conf_gain": positive_conf_gain,
    })

results_df = pd.DataFrame(results_rows)
print(f"Classified {len(results_df)} ROI pairs")
results_df.head()

## 10. Save per-ROI classification results

In [ ]:
classification_results_csv = OUT_DIR / "classification_results.csv"
results_df.to_csv(classification_results_csv, index=False)

print("Saved classification results to:")
print(classification_results_csv)

## 11. Compute semantic-gain metrics

In [ ]:
def safe_mean(series):
    series = pd.Series(series).dropna()
    return float(series.mean()) if len(series) > 0 else None

def safe_median(series):
    series = pd.Series(series).dropna()
    return float(series.median()) if len(series) > 0 else None

if len(results_df) == 0:
    semantic_summary = {
        "num_classified_rois": 0
    }
else:
    small_mask = results_df["area_frac"] < 0.015 if "area_frac" in results_df.columns else pd.Series([False] * len(results_df))

    semantic_summary = {
        "num_classified_rois": int(len(results_df)),
        "mean_video_conf": safe_mean(results_df["video_pred_conf"]),
        "mean_still_conf": safe_mean(results_df["still_pred_conf"]),
        "mean_conf_gain": safe_mean(results_df["conf_gain"]),
        "median_conf_gain": safe_median(results_df["conf_gain"]),
        "positive_conf_gain_rate": float(results_df["positive_conf_gain"].mean()),
        "pred_change_rate": float(results_df["pred_changed"].mean()),
        "mean_video_entropy": safe_mean(results_df["video_entropy"]),
        "mean_still_entropy": safe_mean(results_df["still_entropy"]),
        "mean_entropy_gain": safe_mean(results_df["entropy_gain"]),
        "small_object_mean_conf_gain": safe_mean(results_df.loc[small_mask, "conf_gain"]) if small_mask.any() else None,
        "small_object_count": int(small_mask.sum()) if hasattr(small_mask, "sum") else 0,
    }

print(json.dumps(semantic_summary, indent=2))

## 12. Save the semantic-gain summary

In [ ]:
semantic_gain_summary_json = OUT_DIR / "semantic_gain_summary.json"
with open(semantic_gain_summary_json, "w") as f:
    json.dump(semantic_summary, f, indent=2)

print("Saved semantic summary to:")
print(semantic_gain_summary_json)

## 13. Visualize examples where the still branch helps the most

In [ ]:
import matplotlib.pyplot as plt

if len(results_df) == 0:
    print("No classified rows available for visualization.")
else:
    top_gain_df = results_df.sort_values("conf_gain", ascending=False).head(min(4, len(results_df)))

    plt.figure(figsize=(12, 3 * len(top_gain_df)))
    for i, (_, row) in enumerate(top_gain_df.iterrows(), 1):
        comp_img = np.array(Image.open(row["comp_crop_path"]).convert("RGB"))
        still_img = np.array(Image.open(row["decoded_still_path"]).convert("RGB"))

        plt.subplot(len(top_gain_df), 2, 2 * i - 1)
        plt.imshow(comp_img)
        plt.title(
            "Compressed-video crop\n"
            f'{row["video_pred_label"]} | conf={row["video_pred_conf"]:.3f}'
        )
        plt.axis("off")

        plt.subplot(len(top_gain_df), 2, 2 * i)
        plt.imshow(still_img)
        plt.title(
            "Decoded still crop\n"
            f'{row["still_pred_label"]} | conf={row["still_pred_conf"]:.3f}\n'
            f'conf_gain={row["conf_gain"]:.3f}'
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()

## 14. Coverage and semantic-gain tables

In [ ]:
if len(results_df) == 0:
    print("No results available.")
else:
    overview_df = pd.DataFrame([semantic_summary])
    display(overview_df)

    if "cls_name" in results_df.columns:
        classwise_df = results_df.groupby("cls_name").agg(
            num_rois=("cls_name", "size"),
            mean_video_conf=("video_pred_conf", "mean"),
            mean_still_conf=("still_pred_conf", "mean"),
            mean_conf_gain=("conf_gain", "mean"),
            positive_conf_gain_rate=("positive_conf_gain", "mean"),
            pred_change_rate=("pred_changed", "mean"),
        ).reset_index().sort_values("num_rois", ascending=False)
        display(classwise_df.head(20))

## 15. Save a run summary

In [ ]:
run_summary = {
    "sequence_name": SEQUENCE_NAME,
    "regime": REGIME,
    "roi_still_manifest_csv": str(ROI_STILL_MANIFEST_CSV),
    "classification_results_csv": str(classification_results_csv),
    "semantic_gain_summary_json": str(semantic_gain_summary_json),
    "num_input_rows": int(len(work_df)),
    "num_classified_rois": int(semantic_summary.get("num_classified_rois", 0)),
    "mean_video_conf": semantic_summary.get("mean_video_conf"),
    "mean_still_conf": semantic_summary.get("mean_still_conf"),
    "mean_conf_gain": semantic_summary.get("mean_conf_gain"),
    "positive_conf_gain_rate": semantic_summary.get("positive_conf_gain_rate"),
    "pred_change_rate": semantic_summary.get("pred_change_rate"),
    "small_object_mean_conf_gain": semantic_summary.get("small_object_mean_conf_gain"),
    "mean_entropy_gain": semantic_summary.get("mean_entropy_gain"),
}

run_summary_json = OUT_DIR / "run_summary.json"
with open(run_summary_json, "w") as f:
    json.dump(run_summary, f, indent=2)

print("Saved run summary to:")
print(run_summary_json)
print(json.dumps(run_summary, indent=2))

## 16. What this notebook completed

At this point you now have:
- per-ROI classification results for:
  - compressed-video crops
  - decoded JPEG stills
- per-ROI semantic-gain values
- summary semantic-gain metrics:
  - mean confidence gain
  - positive confidence gain rate
  - prediction change rate
  - small-object mean confidence gain
  - mean entropy gain

## What Notebook 5 should do next

Notebook 5 should combine:
- Notebook 1 video bitrate summary
- Notebook 3 ROI still bitrate summary
- Notebook 4 semantic-gain summary

Then it should compute system-level metrics such as:
- `hybrid_bitrate_mbps`
- `roi_bitrate_share`
- `conf_gain_per_kb`
- matched-budget hybrid-vs-video summaries